In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q torch torchvision scikit-image tqdm pillow matplotlib

import os, torch, torch.nn as nn, torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

BASE = "/content/drive/MyDrive/image_compression_project"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", device)


Mounted at /content/drive
✅ Using device: cuda


In [2]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

train_ds = datasets.CIFAR10(root='data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
print("✅ Train dataset:", len(train_ds))


100%|██████████| 170M/170M [00:04<00:00, 35.1MB/s]


✅ Train dataset: 50000


In [3]:
class VAEGAN_Generator(nn.Module):
    def __init__(self, latent_dim=512):
        super().__init__()
        self.enc_conv = nn.Sequential(
            nn.Conv2d(3,32,4,2,1), nn.ReLU(),
            nn.Conv2d(32,64,4,2,1), nn.ReLU(),
            nn.Conv2d(64,128,4,2,1), nn.ReLU(),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(128*16*16, latent_dim)
        self.fc_logvar = nn.Linear(128*16*16, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, 128*16*16)
        self.dec = nn.Sequential(
            nn.Unflatten(1,(128,16,16)),
            nn.ConvTranspose2d(128,64,4,2,1), nn.ReLU(),
            nn.ConvTranspose2d(64,32,4,2,1), nn.ReLU(),
            nn.ConvTranspose2d(32,3,4,2,1), nn.Sigmoid()
        )
    def reparam(self, mu, logvar):
        std = (0.5 * logvar).exp()
        eps = torch.randn_like(std)
        return mu + eps * std
    def forward(self,x):
        h = self.enc_conv(x)
        mu = self.fc_mu(h); logvar = self.fc_logvar(h)
        z = self.reparam(mu, logvar)
        out = self.dec(self.fc_dec(z))
        return out, mu, logvar

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,64,4,2,1), nn.LeakyReLU(0.2),
            nn.Conv2d(64,128,4,2,1), nn.LeakyReLU(0.2),
            nn.Conv2d(128,256,4,2,1), nn.LeakyReLU(0.2),
            nn.Conv2d(256,1,4,1,0)
        )
    def forward(self,x):
        return self.net(x)

G = VAEGAN_Generator(latent_dim=512).to(device)
D = Discriminator().to(device)
print("✅ Models ready.")


✅ Models ready.


In [4]:
bce = nn.BCEWithLogitsLoss()
mse = nn.MSELoss()
optG = optim.Adam(G.parameters(), lr=1e-4)
optD = optim.Adam(D.parameters(), lr=2e-4)
# Optionally warm-start from VAE:
# G.load_state_dict(torch.load(BASE+"/models/vae_epoch10.pth"), strict=False)


In [5]:
epochs = 12
for epoch in range(epochs):
    G.train(); D.train()
    totalG=0; totalD=0
    for imgs,_ in tqdm(train_loader, desc=f"VAE-GAN Epoch {epoch+1}/{epochs}"):
        imgs = imgs.to(device)
        bs = imgs.size(0)

        # --- Train D ---
        with torch.no_grad():
            fake,_,_ = G(imgs)
        real_logits = D(imgs)
        fake_logits = D(fake.detach())
        real_labels = torch.ones_like(real_logits)
        fake_labels = torch.zeros_like(fake_logits)
        lossD = bce(real_logits, real_labels) + bce(fake_logits, fake_labels)
        optD.zero_grad(); lossD.backward(); optD.step()
        totalD += lossD.item()

        # --- Train G ---
        fake, mu, logvar = G(imgs)
        recon_loss = mse(fake, imgs)
        adv_loss = bce(D(fake), torch.ones_like(D(fake)))
        kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / bs
        lossG = recon_loss + 0.01 * adv_loss + 1e-4 * kl
        optG.zero_grad(); lossG.backward(); optG.step()
        totalG += lossG.item()

    print(f"Epoch {epoch+1}: D={totalD/len(train_loader):.4f}, G={totalG/len(train_loader):.4f}")

torch.save(G.state_dict(), os.path.join(BASE,"models/vaegan_epoch12.pth"))
print("💾 Saved ->", os.path.join(BASE,"models/vaegan_epoch12.pth"))


VAE-GAN Epoch 1/12: 100%|██████████| 782/782 [02:43<00:00,  4.78it/s]


Epoch 1: D=0.7328, G=0.0829


VAE-GAN Epoch 2/12: 100%|██████████| 782/782 [02:49<00:00,  4.61it/s]


Epoch 2: D=0.7933, G=0.0646


VAE-GAN Epoch 3/12: 100%|██████████| 782/782 [02:50<00:00,  4.58it/s]


...

Epoch 10: D=1.2787, G=0.0299


VAE-GAN Epoch 11/12: 100%|██████████| 782/782 [02:50<00:00,  4.59it/s]


Epoch 11: D=1.4351, G=0.0313


VAE-GAN Epoch 12/12: 100%|██████████| 782/782 [02:50<00:00,  4.59it/s]


Epoch 12: D=1.3207, G=0.0289
💾 Saved -> /content/drive/MyDrive/image_compression_project/models/vaegan_epoch12.pth


In [6]:
from torchvision import utils as vutils
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim
import numpy as np

transform = transforms.Compose([transforms.Resize((128,128)), transforms.ToTensor()])
test_ds = datasets.STL10(root='data', split='test', download=True, transform=transform)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)
os.makedirs(os.path.join(BASE,"results/grids"), exist_ok=True)

G.eval()
psnrs, ssims = [], []
for i,(img,_) in enumerate(test_loader):
    if i >= 24: break
    img = img.to(device)
    with torch.no_grad():
        rec,_,_ = G(img)
    orig_np = img.cpu().squeeze(0).permute(1,2,0).numpy()
    rec_np  = rec.cpu().squeeze(0).permute(1,2,0).numpy()
    psnrs.append(psnr(orig_np, rec_np, data_range=1.0))
    ssims.append(ssim(orig_np, rec_np, channel_axis=-1, data_range=1.0, win_size=7))
    vutils.save_image(torch.cat([img.cpu(), rec.cpu()], dim=3),
                      os.path.join(BASE,f"results/grids/vaegan_grid_{i+1:02d}.png"))
print(f"📈 Avg PSNR = {np.mean(psnrs):.2f}, SSIM = {np.mean(ssims):.3f}")


100%|██████████| 2.64G/2.64G [00:49<00:00, 53.2MB/s]


📈 Avg PSNR = 16.90, SSIM = 0.287
